# Metodologia Design Science Research (DSR)

**Etapa de Pesquisa (Peffers et al., 2007):**
### 3. Design e Desenvolvimento (Design and Development)

**Objetivo Acadêmico:** Este notebook trata da ingestão de dados meteorológicos históricos. No rigor do DSR, isso representa a inclusão de variáveis exógenas de controle. A literatura (RODRIGUES et al., 2024) indica que o clima é um fator de variabilidade em serviços de alimentação, influenciando tanto a escolha do cardápio quanto a decisão de consumo dos discentes.


# 01e - Ingestão de Dados Climáticos (API)
Este notebook coleta dados históricos e previsões meteorológicas via Open-Meteo API para Itaquaquecetuba.

In [1]:
# Instalação das bibliotecas necessárias
!pip install openmeteo-requests requests-cache retry-requests numpy pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
from datetime import datetime, timedelta

# 1. Configurações de Período
# Período solicitado: 01/01/2023 até Hoje + 14 dias
start_date = "2023-01-01"
end_date = (datetime.now() + timedelta(days=14)).strftime('%Y-%m-%d')

print(f"🌦️ Iniciando coleta de {start_date} até {end_date}...")

# 2. Configuração da API
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# URL que suporta histórico e forecast
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": -23.48563827, 
    "longitude": -46.34327828,
    "start_date": start_date,
    "end_date": end_date,
    "daily": ["temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "rain_sum", "wind_speed_10m_max"],
    "timezone": "America/Sao_Paulo"
}

# 3. Requisição e Processamento
try:
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    daily = response.Daily()

    daily_data = {
        "data": pd.date_range(
            start=pd.to_datetime(daily.Time(), unit="s", utc=True).tz_convert("America/Sao_Paulo"),
            end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True).tz_convert("America/Sao_Paulo"),
            freq=pd.Timedelta(seconds=daily.Interval()),
            inclusive="left"
        ).strftime('%Y-%m-%d'),
        "temp_media": daily.Variables(0).ValuesAsNumpy().round(2),
        "temp_max": daily.Variables(1).ValuesAsNumpy().round(2),
        "temp_min": daily.Variables(2).ValuesAsNumpy().round(2),
        "chuva_soma": daily.Variables(3).ValuesAsNumpy().round(2),
        "vento_max": daily.Variables(4).ValuesAsNumpy().round(2)
    }

    df_clima = pd.DataFrame(data=daily_data)

    # 4. Exportação
    saida_csv = "../data/clima_consolidado.csv"
    df_clima.to_csv(saida_csv, index=False)

    print(f"\n✅ SUCESSO! {len(df_clima)} registros coletados.")
    print(f"📁 Arquivo salvo em: {saida_csv}")
    # Sumário detalhado abaixo

except Exception as e:
    print(f"❌ Erro ao buscar dados: {e}")


🌦️ Iniciando coleta de 2023-01-01 até 2026-03-23...



✅ SUCESSO! 1178 registros coletados.
📁 Arquivo salvo em: ../data/clima_consolidado.csv


# 2. Sumário da Ingestão de Clima (API)
Resumo das variáveis meteorológicas e auditoria de cobertura.

In [3]:
# Sumário Simplificado (Texto e Tabela Detalhada)
import pandas as pd
df_temp = df_clima.copy()
df_temp['data'] = pd.to_datetime(df_temp['data'])
total_registros = len(df_temp)
primeira_data = df_temp['data'].min().strftime('%d/%m/%Y')
ultima_data = df_temp['data'].max().strftime('%d/%m/%Y')

# Extremos para Auditoria
max_temp = df_temp['temp_max'].max()
min_temp = df_temp['temp_min'].min()
max_chuva = df_temp['chuva_soma'].max()

print(f"--- RELATÓRIO DE INGESTÃO (CLIMA) ---")
print(f"Total de Registros Coletados: {total_registros}")
print(f"Período: {primeira_data} até {ultima_data}")
print(f"Temperatura Máxima no Período: {max_temp:.1f}°C")
print(f"Temperatura Mínima no Período: {min_temp:.1f}°C")
print(f"Maior Volume de Chuva em 24h: {max_chuva:.1f}mm")

# 1. Visão Detalhada: Últimos 30 dias (Foco em Auditoria Recente)
print("\nDetalhamento Recente (Últimos 30 registros):")
detalhe_clima = df_temp.tail(30)[['data', 'temp_max', 'temp_min', 'chuva_soma']].copy()
detalhe_clima.columns = ['Data', 'Máxima (°C)', 'Mínima (°C)', 'Chuva (mm)']
detalhe_clima['Data'] = detalhe_clima['Data'].dt.strftime('%d/%m/%Y')

# Estilizar com gradientes para destacar extremos
display(detalhe_clima.style.background_gradient(cmap='YlOrRd', subset=['Máxima (°C)'])
                    .background_gradient(cmap='Blues', subset=['Chuva (mm)']))

# 2. Visão Mensal Rápida (Média de Temperatura e Soma de Chuva)
df_temp['ano_mes'] = df_temp['data'].dt.strftime('%Y-%m')
mensal = df_temp.groupby('ano_mes').agg({'temp_media': 'mean', 'chuva_soma': 'sum'}).reset_index()
mensal.columns = ['Mês', 'Temp. Média (°C)', 'Chuva Total (mm)']
print("\nConsolidado Mensal:")
print(mensal.tail(12).to_string(index=False)) # Mostrar o último ano

--- RELATÓRIO DE INGESTÃO (CLIMA) ---
Total de Registros Coletados: 1178
Período: 01/01/2023 até 23/03/2026
Temperatura Máxima no Período: 37.8°C
Temperatura Mínima no Período: 3.5°C
Maior Volume de Chuva em 24h: 79.2mm

Detalhamento Recente (Últimos 30 registros):


,Data,Máxima (°C),Mínima (°C),Chuva (mm)
1148,22/02/2026,26.700001,20.299999,0.000000
1149,23/02/2026,26.500000,20.650000,0.000000
1150,24/02/2026,26.900000,20.400000,0.000000
1151,25/02/2026,29.350000,21.100000,0.000000
1152,26/02/2026,24.799999,21.250000,0.000000
1153,27/02/2026,23.250000,18.350000,0.000000
1154,28/02/2026,25.000000,16.299999,0.000000
1155,01/03/2026,24.450001,17.150000,0.000000
1156,02/03/2026,25.900000,15.800000,0.000000
1157,03/03/2026,27.049999,15.500000,0.000000



Consolidado Mensal:
    Mês  Temp. Média (°C)  Chuva Total (mm)
2025-04         20.358334         11.300000
2025-05         18.316452          0.300000
2025-06         17.277334          7.500000
2025-07         15.608065          1.600000
2025-08         17.354839          1.000000
2025-09         19.723667          0.100000
2025-10         19.985161         15.900000
2025-11         20.971998         11.400000
2025-12         24.394838         41.599998
2026-01         23.123869          7.600000
2026-02         23.520000          9.000000
2026-03         21.496088         53.099998
